# Project Overview

This notebook demonstrates the process of fine-tuning OpenAI's Whisper model on Persian speech data.  
The project leverages Mozilla's Common Voice dataset for training and evaluation, combined with Hugging Face's `transformers` and `datasets` libraries for model handling and dataset management.

The primary goal is to teach Whisper to transcribe Persian audio more accurately by exposing it to custom data through supervised fine-tuning.

---


# Install Required Packages

This cell installs all the necessary Python libraries for the fine-tuning pipeline.  
The packages include Hugging Face's `datasets` for loading audio datasets, `transformers` for handling pre-trained models, and `torch` for deep learning.  
Additional tools like `evaluate`, `jiwer`, `tensorboard`, and `gradio` are included for training, monitoring, and serving the model.

Make sure to run this cell before executing any other part of the notebook!

---

In [ ]:
# !pip install datasets[audio] transformers "gcsfs<2025" torch accelerate evaluate hf_xet librosa jiwer tensorboard tensorboardX gradio fsspec

# Hugging Face Authentication

This cell logs into your Hugging Face account using a personal access token.  
Logging in is required to download pre-trained models, push fine-tuned models to the Hugging Face Hub, and access gated resources.

Make sure to replace `"YOUR_HF_TOKEN"` with your actual token from your Hugging Face account.  
You can generate one here: https://huggingface.co/settings/tokens

---

In [ ]:
# login to huggingface
from huggingface_hub import login

HUGGINGFACE_TOKEN = "YOUR_HF_TOKEN"

login(HUGGINGFACE_TOKEN)

# Loading the Common Voice Dataset

This cell loads the Mozilla Common Voice dataset for Persian (`fa`) using Hugging Face's `datasets` library.  
It combines the `train` and `validation` splits for training and uses the official `test` split for evaluation.  
The `trust_remote_code=True` argument allows the dataset to load any custom code provided by the dataset authors.

The dataset will be structured as a `DatasetDict` object with two keys: `"train"` and `"test"`.  
This step is essential for preparing audio-text pairs for the fine-tuning process.

---

In [ ]:
from datasets import load_dataset, DatasetDict

common_voice = DatasetDict()
common_voice["train"] = load_dataset(
    "mozilla-foundation/common_voice_17_0", "fa", split="train+validation", trust_remote_code=True
)
common_voice["test"] = load_dataset(
    "mozilla-foundation/common_voice_17_0", "fa", split="test", trust_remote_code=True
)

print(common_voice)

# Cleaning Dataset Columns

This cell removes unnecessary metadata columns from the Common Voice dataset,  
keeping only the essential fields for training: typically `audio` and `sentence`.

---

In [ ]:
common_voice = common_voice.remove_columns(["accent", "variant", "age", "client_id", "down_votes", "gender", "locale", "path", "segment", "up_votes"])

In [ ]:
print(common_voice)

# Initializing Feature Extractor

This cell loads the pre-trained Whisper feature extractor from Hugging Face.  
It processes raw audio into input features compatible with the Whisper model.

---

In [ ]:
from transformers import WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")

# Initializing Tokenizer

This cell loads the Whisper tokenizer pre-trained for the `Persian` language,  
configured for the transcription task. It converts text to token IDs and vice versa for training.

---


In [ ]:
from transformers import WhisperTokenizer

tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", language="Persian", task="transcribe")

# Initializing Processor

This cell loads the Whisper processor,  
which combines both the feature extractor and tokenizer into a single interface for easier data preprocessing.

---

In [ ]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="Persian", task="transcribe")

In [ ]:
print(common_voice["train"][0])

# Casting Audio Column

This cell converts the `audio` column to a standardized format with a fixed `sampling_rate` of 16kHz,  
ensuring compatibility with the Whisper feature extractor.

---

In [ ]:
from datasets import Audio

common_voice = common_voice.cast_column("audio", Audio(sampling_rate=16000))
print(common_voice["train"][0])

# Preparing Dataset

This cell defines a function to process and prepare the dataset for fine-tuning. It converts the audio data into the required input features using the Whisper feature extractor, and tokenizes the corresponding sentences into label IDs using the Whisper tokenizer. The dataset is then mapped and processed in batches.

----

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]

    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

common_voice = common_voice.map(prepare_dataset, remove_columns=common_voice.column_names["train"], num_proc=1)

# Loading and Configuring the Model

This cell loads the pre-trained Whisper model (`whisper-small`) from Hugging Face's model hub. The model's generation configuration is then updated to set the language to Persian and the task to transcription. Additionally, the forced decoder IDs are set to `None` to allow the model to generate text freely.

----

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

model.generation_config.language = "persian"
model.generation_config.task = "transcribe"

model.generation_config.forced_decoder_ids = None

# Defining Data Collator

This cell defines a custom data collator class, `DataCollatorSpeechSeq2SeqWithPadding`, which is responsible for padding and processing the input features and labels in a way that is compatible with Whisper’s architecture. The collator splits inputs and labels, applying different padding strategies for each. It pads the audio inputs using the feature extractor and pads the tokenized labels to the maximum length. The labels are then adjusted to ignore padding tokens during loss calculation. Additionally, any beginning-of-sequence tokens are removed if they were added in a previous tokenization step.

----

In [ ]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch


# Initializing Data Collator

This cell initializes the `DataCollatorSpeechSeq2SeqWithPadding` with the Whisper processor and the model's `decoder_start_token_id`. This prepares the data collator to handle padding and batching of both input features and labels during training.

----

In [ ]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

# Loading Evaluation Metric

This cell loads the Word Error Rate (WER) metric from the `evaluate` library, which will be used to assess the performance of the model during training and evaluation.

----

In [ ]:
import evaluate

metric = evaluate.load("wer")

# Defining Metrics Calculation

This cell defines the `compute_metrics` function, which calculates the Word Error Rate (WER) metric. It takes the model's predictions and the corresponding label IDs, replaces any padding tokens (`-100`) with the `pad_token_id`, and decodes the predicted and reference sequences. Finally, the WER is computed by comparing the decoded predictions to the reference labels.

----

In [ ]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

# Setting Training Arguments

This cell defines the training arguments for fine-tuning the Whisper model using `Seq2SeqTrainingArguments`. Key parameters include batch size, learning rate, gradient accumulation, and training steps. It also enables features like gradient checkpointing for memory optimization, mixed precision (FP16) training, and evaluation strategy. Additionally, the best model is saved based on the WER metric, and TensorBoard is used for logging training progress.

----

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-fa",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    warmup_steps=500,
    max_steps=5000,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=1000,
    eval_steps=1000,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=True,
)

# Initializing the Trainer

This cell initializes the `Seq2SeqTrainer` from the Hugging Face Transformers library, setting up the training loop for fine-tuning the Whisper model. It specifies the model, datasets (training and evaluation), data collator, and metrics computation function. The training arguments and tokenizer are also passed to ensure proper configuration for training.

----

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=common_voice["train"],
    eval_dataset=common_voice["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

# Starting the Training Process

This cell starts the training process using the `train()` method of the `Seq2SeqTrainer`. It begins fine-tuning the Whisper model on the provided training dataset, using the specified training arguments, data collator, and metrics.

----

In [ ]:
trainer.train()